
# blob > 1행컬럼 2행부터데이터 맞춰서 bronze 노트북으로 넘기기
--------
- 입력: blob의 7개 부두 x 2개 스냅샷 = 14개 파일 (xlsx / xls(html) / xml)
- 출력: 파일 단위 리스트
        [
          {"file": "terminal_1_schedule_20260626_10.xls",
           "terminal_no": 1,
           "df": <DataFrame: 1행=컬럼명, 2행~=데이터>},
          ...
        ]
- 이 결과를 팀원이 받아서 snapshot_time / hash 등 추가 컬럼을 붙여 bronze로 적재함.
- 컬럼명 매핑/표준화는 silver 단계에서 처리하므로 여기서는 원본 컬럼명을 그대로 보존한다.
- terminal_6(xml)은 Nexacro Platform 데이터셋 형식으로, ColumnInfo에 정의된
  컬럼 순서를 기준으로 Row를 채운다 (read_nexacro_xml 참고).
--------


### 0. 패키지 및 라이브러리 불러오기 

In [0]:
%pip install openpyxl html5lib lxml beautifulsoup4 xlrd fsspec

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.2/112.2 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.6/96.6 kB 11.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 13.1 MB/s eta 0:00:00
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import os
import re
import pandas as pd
import numpy as np
import hashlib
import json
import xml.etree.ElementTree as ET
from io import StringIO
from openpyxl import load_workbook

In [0]:
# ==========
# 1. 파일 읽기
# ===========

# blob storage 컨테이너 이름(사용할 컨테이너 이름으로 바꾸면 됩니다.)
container = "raw-schedule"

# 스토리지 계정(고정)
storage_account = "dt4team3storage"

blob_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/"

files = dbutils.fs.ls(blob_path)
for f in files:
    print(f.name, f.size)


terminal_1_schedule_20260626_10.xls 23267
terminal_1_schedule_20260626_11.xls 23267
terminal_2_schedule_20260626_10.xls 22762
terminal_2_schedule_20260626_11.xls 22763
terminal_3_schedule_20260626_10.xlsx 6852
terminal_3_schedule_20260626_11.xlsx 6856
terminal_4_schedule_20260626_10.xls 26230
terminal_4_schedule_20260626_11.xls 26230
terminal_5_schedule_20260626_10.xlsx 27756
terminal_5_schedule_20260626_11.xlsx 27756
terminal_6_schedule_20260626_10.xml 24965
terminal_6_schedule_20260626_11.xml 24965
terminal_7_schedule_20260626_10.xlsx 45590
terminal_7_schedule_20260626_11.xlsx 45591


### 1. 함수

In [0]:
#  -----------------------------------------------------------
# 0) blob -> workspace 로컬 복사
#    pandas/ET는 wasbs:// 경로를 직접 못 읽으므로, 로컬 디스크로 먼저 내려받는다.
#    팀 공유 작업 폴더(/Workspace/Shared/...)를 사용.
# -----------------------------------------------------------
def copy_blob_file_to_local(blob_file_path):
    '''
    Blob 파일을 Workspace 공유 폴더로 복사하는 함수
    '''
    workspace_tmp_dir = "/Workspace/Shared/busan_port_project/01_schedule_pipeline/tmp_excel"
    dbutils.fs.mkdirs(workspace_tmp_dir)
 
    local_path = f"{workspace_tmp_dir}/{os.path.basename(blob_file_path)}"
    dbutils.fs.cp(blob_file_path, f"file:{local_path}")
 
    return local_path

In [0]:
# -----------------------------------------------------------
# 1) 부두번호 추출
# -----------------------------------------------------------
def extract_terminal_no(file_name: str) -> int:
    """
    'terminal_3_schedule_20260626_10.xlsx' -> 3
    파일명 규칙이 깨지면 None -> 호출부에서 에러로 처리
    """
    m = re.search(r"terminal_(\d+)_", file_name)
    return int(m.group(1)) if m else None

In [0]:
# -----------------------------------------------------------
# 2) 부두별 헤더 규칙
#    no_header   : 원본에 헤더 없음 -> col_1, col_2... 부여, 0행부터 데이터
#    header_row  : header_row번째 행(0-base)이 헤더, 그 다음행부터 데이터
#    xml_custom  : xml(Nexacro 데이터셋) 전용, read_nexacro_xml에서 ColumnInfo 기준 처리
# -----------------------------------------------------------
TERMINAL_HEADER_RULES = {
    1: {"mode": "no_header"},                      # xls(html)
    2: {"mode": "no_header"},                      # xls(html)
    3: {"mode": "header_row", "header_row": 1},    # xlsx: 1행 제목 / 2행 헤더
    4: {"mode": "no_header"},                      # xls(html)
    5: {"mode": "header_row", "header_row": 0},    # xlsx: 1행 헤더
    6: {"mode": "xml_custom"},                     # xml - Nexacro ColumnInfo 기반, read_nexacro_xml에서 이미 헤더 확정
    7: {"mode": "header_row", "header_row": 0},    # xlsx: 1행 헤더
}

In [0]:
# -----------------------------------------------------------
# 3) 확장자별 원시 데이터 읽기 (헤더 처리 없이 전체 그대로)
# -----------------------------------------------------------
def read_raw_table(local_path: str) -> pd.DataFrame:
    file_name = os.path.basename(local_path)
 
    if file_name.endswith(".xlsx"):
        return pd.read_excel(local_path, header=None, engine="openpyxl")
 
    elif file_name.endswith(".xls"):
        with open(local_path, "rb") as f:
            content = f.read()
        content = content.replace(b"udf-8", b"utf-8").replace(b"UDF-8", b"UTF-8")
        try:
            html_text = content.decode("utf-8")
        except UnicodeDecodeError:
            html_text = content.decode("cp949", errors="replace")
        tables = pd.read_html(StringIO(html_text), header=None)
        return tables[0]
 
    elif file_name.endswith(".xml"):
        return read_nexacro_xml(local_path)
 
    else:
        raise ValueError(f"지원하지 않는 확장자: {file_name}")

In [0]:
def read_nexacro_xml(local_path: str) -> pd.DataFrame:
    """
    Nexacro Platform 데이터셋 XML 전용 파서.
    구조:
        <Root xmlns="http://www.nexacroplatform.com/platform/dataset...">
          <Dataset id="output1">
            <ColumnInfo>
              <Column id="plvVoy" .../>
              <Column id="plvShiftvan" .../>
              ...
            </ColumnInfo>
            <Rows>
              <Row>
                <Col id="plvVoy">001</Col>
                ...
              </Row>
            </Rows>
          </Dataset>
        </Root>
 
    ColumnInfo에서 컬럼 순서를 먼저 확정한 뒤, 그 순서대로 Row를 채운다.
    -> 특정 행에 일부 Col 태그가 비어 있어도(예: 옵션값 없음) 컬럼이 밀리지 않는다.
    """
    import xml.etree.ElementTree as ET
 
    tree = ET.parse(local_path)
    root = tree.getroot()
 
    def clean(tag):
        return tag.split("}")[-1]
 
    # 1) ColumnInfo에서 컬럼 순서 확정
    column_order = []
    for elem in root.iter():
        if clean(elem.tag) == "ColumnInfo":
            for col in elem:
                if clean(col.tag) == "Column":
                    col_id = col.attrib.get("id")
                    if col_id:
                        column_order.append(col_id)
            break  # 첫 ColumnInfo만 사용 (Dataset이 여러 개일 경우 대비)
 
    if not column_order:
        raise ValueError("ColumnInfo에서 컬럼 정의를 찾지 못했습니다.")
 
    # 2) Row 단위로 값 채우기 (컬럼 순서 고정)
    rows = []
    for elem in root.iter():
        if clean(elem.tag) == "Row":
            row = {col_id: None for col_id in column_order}
            for child in elem:
                col_id = child.attrib.get("id")
                if col_id in row:
                    row[col_id] = child.text
            rows.append(row)
 
    # 3) 컬럼 순서를 명시적으로 지정해 DataFrame 생성
    df = pd.DataFrame(rows, columns=column_order)
    return df

In [0]:
# -----------------------------------------------------------
# 4) 부두 규칙에 따라 "1행=컬럼명, 2행~=데이터"로 정제
# -----------------------------------------------------------
def apply_header_rule(raw_df: pd.DataFrame, terminal_no: int) -> pd.DataFrame:
    rule = TERMINAL_HEADER_RULES.get(terminal_no)
    if rule is None:
        raise ValueError(f"terminal_{terminal_no}에 대한 헤더 규칙이 없습니다.")
 
    mode = rule["mode"]
 
    if mode == "no_header":
        df = raw_df.copy()
        df.columns = [f"col_{i+1}" for i in range(df.shape[1])]
        return df.reset_index(drop=True)
 
    elif mode == "header_row":
        header_idx = rule["header_row"]
        header = raw_df.iloc[header_idx]
        df = raw_df.iloc[header_idx + 1:].copy()
        df.columns = header.values
        return df.reset_index(drop=True)
 
    elif mode == "xml_custom":
        # xml(Nexacro)은 read_raw_table -> read_nexacro_xml에서
        # ColumnInfo 기준으로 컬럼 순서를 이미 확정해 DataFrame을 만든 상태.
        # -> 추가 헤더 처리 불필요, 그대로 반환.
        return raw_df.reset_index(drop=True)
 
    else:
        raise ValueError(f"알 수 없는 모드: {mode}")
 
 

In [0]:
# -----------------------------------------------------------
# 5) 파일 1개 -> 정제 결과 1건 (dict)
# -----------------------------------------------------------
def process_one_file(local_path: str) -> dict:
    file_name = os.path.basename(local_path)
    terminal_no = extract_terminal_no(file_name)
    if terminal_no is None:
        raise ValueError(f"파일명에서 부두번호를 추출할 수 없습니다: {file_name}")
 
    raw_df = read_raw_table(local_path)
    clean_df = apply_header_rule(raw_df, terminal_no)
 
    return {
        "file": file_name,
        "terminal_no": terminal_no,
        "df": clean_df,
    }

In [0]:
# -----------------------------------------------------------
# 6) 전체 파일 처리 -> 파일 단위 결과 리스트
#    (팀원이 이 리스트를 받아서 추가 컬럼을 붙임)
# -----------------------------------------------------------
def build_bronze_inputs(files) -> list:
    """
    files: dbutils.fs.ls() 결과 리스트
    반환: [{"file":, "terminal_no":, "df":}, ...]  (실패한 파일은 별도 errors 리스트로 출력)
    """
    file_paths = [f.path for f in files if f.name.endswith((".xlsx", ".xls", ".xml"))]
 
    results = []
    errors = []
 
    for path in file_paths:
        try:
            local_path = copy_blob_file_to_local(path)  
            result = process_one_file(local_path)
            results.append(result)
        except Exception as e:
            errors.append({"file": os.path.basename(path), "error": str(e)})
 
    if errors:
        print(f"[경고] {len(errors)}개 파일 처리 실패:")
        for e in errors:
            print(" -", e)
 
    return results

In [0]:
# -----------------------------------------------------------
# 7) 정제 결과를 Parquet으로 저장 + manifest 작성
#    -> %run 전역노출 없이, 팀원이 경로/manifest만 보고 읽어가는 방식
# -----------------------------------------------------------
BRONZE_STAGING_DIR = "/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging"
 
 
def save_bronze_inputs_to_staging(bronze_inputs: list, staging_dir: str = BRONZE_STAGING_DIR) -> dict:
    """
    bronze_inputs(= build_bronze_inputs() 결과)를 부두/스냅샷 파일 단위로 Parquet 저장.
    각 파일의 컬럼명/순서/dtype은 원본 그대로 보존된다 (모든 컬럼을 string으로 캐스팅해서
    저장 - 타입 캐스팅은 silver 단계에서 처리하므로 bronze에서는 원본 표현을 그대로 둔다).
 
    주의: dbutils.fs.*(DBFS)와 순수 Python(os/pandas)이 보는 /Workspace/... 경로는
    서로 다른 파일시스템 레이어로 동작할 수 있다 (DBFS에는 쓰였지만 실제 Workspace
    파일에는 반영되지 않는 현상 확인됨). 따라서 이 함수는 dbutils.fs를 전혀 쓰지 않고
    os/shutil만으로 직접 써서, Workspace UI와 pandas.read_parquet() 양쪽에서
    동일하게 보이도록 한다.
 
    manifest.json에는 팀원이 읽어야 할 파일 목록과 메타정보를 기록한다.
    반환값: manifest dict (저장 직후 바로 확인할 수 있도록)
    """
    import json
 
    os.makedirs(staging_dir, exist_ok=True)
 
    manifest_entries = []
 
    for item in bronze_inputs:
        file_name = item["file"]
        terminal_no = item["terminal_no"]
        df = item["df"]
 
        # 원본 컬럼 표현을 그대로 보존하기 위해 전부 string으로 캐스팅 후 저장
        # (parquet은 컬럼별 단일 dtype을 요구하므로, object/mixed 타입 충돌을 막기 위함)
        df_to_save = df.astype(str)
 
        parquet_file_name = os.path.splitext(file_name)[0] + ".parquet"
        final_path = f"{staging_dir}/{parquet_file_name}"
        df_to_save.to_parquet(final_path, index=False)  # staging_dir에 직접 저장 (중간 복사 단계 제거)
 
        manifest_entries.append({
            "file": file_name,
            "parquet_file": parquet_file_name,
            "terminal_no": terminal_no,
            "rows": df.shape[0],
            "cols": df.shape[1],
            "columns": df.columns.tolist(),
        })
 
    manifest = {
        "staging_dir": staging_dir,
        "file_count": len(manifest_entries),
        "entries": manifest_entries,
    }
 
    manifest_path = f"{staging_dir}/manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
 
    print(f"저장 완료: {len(manifest_entries)}개 파일 -> {staging_dir}")
    print(f"manifest.json: {manifest_path}")
 
    return manifest

### 3. 지나님께 넘기기

In [0]:
# -----------------------------------------------------------
# 7) 정제 결과를 Parquet으로 저장 + manifest 작성
#    -> %run 전역노출 없이, 팀원이 경로/manifest만 보고 읽어가는 방식
# -----------------------------------------------------------

BRONZE_STAGING_DIR = "/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging"

In [0]:
# try1
def save_bronze_inputs_to_staging(bronze_inputs: list, staging_dir: str = BRONZE_STAGING_DIR) -> dict:
    """
    bronze_inputs(= build_bronze_inputs() 결과)를 부두/스냅샷 파일 단위로 Parquet 저장.
    각 파일의 컬럼명/순서/dtype은 원본 그대로 보존된다 (모든 컬럼을 string으로 캐스팅해서
    저장 - 타입 캐스팅은 silver 단계에서 처리하므로 bronze에서는 원본 표현을 그대로 둔다).
 
    manifest.json에는 팀원이 읽어야 할 파일 목록과 메타정보를 기록한다.
    반환값: manifest dict (저장 직후 바로 확인할 수 있도록)
    """
    dbutils.fs.mkdirs(staging_dir)
 
    manifest_entries = []
 
    # 로컬 임시 작업공간: copy_blob_file_to_local()에서 쓰는 폴더를 그대로 재사용
    # (/tmp는 클러스터에서 쓰기 권한이 막혀있을 수 있어 피함)
    local_tmp_dir = "/Workspace/Shared/busan_port_project/01_schedule_pipeline/tmp_excel"
    dbutils.fs.mkdirs(local_tmp_dir)
 
    for item in bronze_inputs:
        file_name = item["file"]
        terminal_no = item["terminal_no"]
        df = item["df"]
 
        # 원본 컬럼 표현을 그대로 보존하기 위해 전부 string으로 캐스팅 후 저장
        # (parquet은 컬럼별 단일 dtype을 요구하므로, object/mixed 타입 충돌을 막기 위함)
        df_to_save = df.astype(str)
 
        parquet_file_name = os.path.splitext(file_name)[0] + ".parquet"
        local_parquet_path = f"{local_tmp_dir}/{parquet_file_name}"
        df_to_save.to_parquet(local_parquet_path, index=False)
 
        final_path = f"{staging_dir}/{parquet_file_name}"
        dbutils.fs.cp(f"file:{local_parquet_path}", final_path)
 
        manifest_entries.append({
            "file": file_name,
            "parquet_file": parquet_file_name,
            "terminal_no": terminal_no,
            "rows": df.shape[0],
            "cols": df.shape[1],
            "columns": df.columns.tolist(),
        })
 
    manifest = {
        "staging_dir": staging_dir,
        "file_count": len(manifest_entries),
        "entries": manifest_entries,
    }
 
    local_manifest_path = f"{local_tmp_dir}/manifest.json"
    with open(local_manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    dbutils.fs.cp(f"file:{local_manifest_path}", f"{staging_dir}/manifest.json")
 
    print(f"저장 완료: {len(manifest_entries)}개 파일 -> {staging_dir}")
    print(f"manifest.json: {staging_dir}/manifest.json")
 
    return manifest

In [0]:

# =====================================================
# 진단: bronze_staging 저장이 왜 안 됐는지 확인
# =====================================================

staging_dir = "/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging"
local_tmp_dir = "/Workspace/Shared/busan_port_project/01_schedule_pipeline/tmp_excel"

print("=== 1. bronze_staging 폴더 존재 여부 + 내용물 ===")
try:
    contents = dbutils.fs.ls(staging_dir)
    if contents:
        for f in contents:
            print(" -", f.name, f.size)
    else:
        print("폴더는 있지만 내용물이 비어있음")
except Exception as e:
    print("폴더 자체가 없음 또는 접근 에러:", e)

print()
print("=== 2. tmp_excel 폴더 내용물 (parquet 파일이 실제로 만들어졌는지) ===")
try:
    contents = dbutils.fs.ls(local_tmp_dir)
    parquet_files = [f for f in contents if f.name.endswith(".parquet")]
    json_files = [f for f in contents if f.name.endswith(".json")]
    print(f"parquet 파일 개수: {len(parquet_files)}")
    for f in parquet_files:
        print("  -", f.name, f.size)
    print(f"json 파일 개수: {len(json_files)}")
    for f in json_files:
        print("  -", f.name, f.size)
except Exception as e:
    print("에러:", e)

print()
print("=== 3. dbutils.fs.cp 단독 테스트 (한 파일만 직접 복사 시도) ===")
if 'bronze_inputs' in dir():
    try:
        sample_item = bronze_inputs[0]
        sample_df = sample_item["df"].astype(str)
        sample_name = os.path.splitext(sample_item["file"])[0] + "_TEST.parquet"

        local_test_path = f"{local_tmp_dir}/{sample_name}"
        sample_df.to_parquet(local_test_path, index=False)
        print(f"로컬 저장 성공: {local_test_path}")

        dbutils.fs.mkdirs(staging_dir)
        print(f"mkdirs 호출 완료: {staging_dir}")

        final_test_path = f"{staging_dir}/{sample_name}"
        dbutils.fs.cp(f"file:{local_test_path}", final_test_path)
        print(f"cp 성공: {final_test_path}")

        # 복사 확인
        check = dbutils.fs.ls(staging_dir)
        print("복사 후 staging_dir 내용:", [f.name for f in check])

    except Exception as e:
        print("여기서 에러 발생! ↓↓↓")
        print(type(e).__name__, ":", e)
else:
    print("bronze_inputs 변수가 없습니다. 먼저 build_bronze_inputs(files)를 실행하세요.")

=== 1. bronze_staging 폴더 존재 여부 + 내용물 ===
폴더 자체가 없음 또는 접근 에러: An error occurred while calling o490.ls.
: java.io.FileNotFoundException: No such file or directory /Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging
	at com.databricks.backend.daemon.data.client.DBFSV2.$anonfun$listStatus$2(DatabricksFileSystemV2.scala:193)
	at com.databricks.s3a.S3AExceptionUtils$.convertAWSExceptionToJavaIOException(DatabricksStreamUtils.scala:66)
	at com.databricks.backend.daemon.data.client.DBFSV2.$anonfun$listStatus$1(DatabricksFileSystemV2.scala:173)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:512)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:621)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:646)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:150)
	at scala.util.Dyn

In [0]:
# =====================================================
# save_bronze_inputs_to_staging 직접 호출 + 진행상황 추적
# (몇 번째 파일에서 멈추는지 확인용)
# =====================================================

staging_dir = "/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging"

try:
    manifest = save_bronze_inputs_to_staging(bronze_inputs, staging_dir=staging_dir)
    print("함수 정상 종료. manifest entries 개수:", len(manifest["entries"]))
except Exception as e:
    print("함수 실행 중 에러 발생!")
    print(type(e).__name__, ":", e)
    import traceback
    traceback.print_exc()

print()
print("=== 실행 직후 bronze_staging 폴더 실제 내용 ===")
try:
    contents = dbutils.fs.ls(staging_dir)
    for f in contents:
        print(" -", f.name, f.size)
    print(f"총 {len(contents)}개 항목 (기대값: parquet 14개 + manifest.json 1개 = 15개)")
except Exception as e:
    print("ls 에러:", e)

저장 완료: 14개 파일 -> /Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging
manifest.json: /Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging/manifest.json
함수 정상 종료. manifest entries 개수: 14

=== 실행 직후 bronze_staging 폴더 실제 내용 ===
 - manifest.json 7738
 - terminal_1_schedule_20260626_10.parquet 10600
 - terminal_1_schedule_20260626_10_TEST.parquet 10600
 - terminal_1_schedule_20260626_11.parquet 10601
 - terminal_2_schedule_20260626_10.parquet 13152
 - terminal_2_schedule_20260626_11.parquet 13156
 - terminal_3_schedule_20260626_10.parquet 14655
 - terminal_3_schedule_20260626_11.parquet 14662
 - terminal_4_schedule_20260626_10.parquet 11710
 - terminal_4_schedule_20260626_11.parquet 11710
 - terminal_5_schedule_20260626_10.parquet 9616
 - terminal_5_schedule_20260626_11.parquet 9616
 - terminal_6_schedule_20260626_10.parquet 15395
 - terminal_6_schedule_20260626_11.parquet 15395
 - terminal_7_schedule_20260626_10.parquet 13883
 - terminal_7_schedule_2

In [0]:
print(len(bronze_inputs))

14


In [0]:
# 테스트로 만들었던 잔여 파일 제거
staging_dir = "/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging"
dbutils.fs.rm(f"{staging_dir}/terminal_1_schedule_20260626_10_TEST.parquet")

print("정리 완료. 최종 내용:")
for f in dbutils.fs.ls(staging_dir):
    print(" -", f.name, f.size)

정리 완료. 최종 내용:
 - manifest.json 7738
 - terminal_1_schedule_20260626_10.parquet 10600
 - terminal_1_schedule_20260626_11.parquet 10601
 - terminal_2_schedule_20260626_10.parquet 13152
 - terminal_2_schedule_20260626_11.parquet 13156
 - terminal_3_schedule_20260626_10.parquet 14655
 - terminal_3_schedule_20260626_11.parquet 14662
 - terminal_4_schedule_20260626_10.parquet 11710
 - terminal_4_schedule_20260626_11.parquet 11710
 - terminal_5_schedule_20260626_10.parquet 9616
 - terminal_5_schedule_20260626_11.parquet 9616
 - terminal_6_schedule_20260626_10.parquet 15395
 - terminal_6_schedule_20260626_11.parquet 15395
 - terminal_7_schedule_20260626_10.parquet 13883
 - terminal_7_schedule_20260626_11.parquet 13891


In [0]:
# =====================================================
# bronze_staging이 실제로 어떤 파일시스템에 있는지 확인
# =====================================================

staging_dir = "/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging"

print("=== 1. dbutils.fs.ls 결과 (전체 path 포함) ===")
for f in dbutils.fs.ls(staging_dir):
    print(f.path)

print()
print("=== 2. 상위 폴더(01_schedule_pipeline) 전체 목록 ===")
parent_dir = "/Workspace/Shared/busan_port_project/01_schedule_pipeline"
for f in dbutils.fs.ls(parent_dir):
    print(f.path, "| isDir:", f.isDir() if hasattr(f, "isDir") else "?")

print()
print("=== 3. Python os 모듈로도 직접 확인 (실제 로컬 파일시스템 기준) ===")
import os
try:
    print(os.listdir(parent_dir))
except Exception as e:
    print("os.listdir 에러:", e)

=== 1. dbutils.fs.ls 결과 (전체 path 포함) ===
dbfs:/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging/manifest.json
dbfs:/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging/terminal_1_schedule_20260626_10.parquet
dbfs:/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging/terminal_1_schedule_20260626_11.parquet
dbfs:/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging/terminal_2_schedule_20260626_10.parquet
dbfs:/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging/terminal_2_schedule_20260626_11.parquet
dbfs:/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging/terminal_3_schedule_20260626_10.parquet
dbfs:/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging/terminal_3_schedule_20260626_11.parquet
dbfs:/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging/terminal_4_schedule_20260626_10.parquet
dbfs:/Workspace/Shared/busan_port_project

In [0]:
import os
import time

parent_dir = "/Workspace/Shared/busan_port_project/01_schedule_pipeline"

print("즉시 확인:", os.listdir(parent_dir))

print("5초 대기 후 재확인...")
time.sleep(5)
print("5초 후:", os.listdir(parent_dir))

print("추가 10초 대기 후 재확인...")
time.sleep(10)
print("15초 후:", os.listdir(parent_dir))

즉시 확인: ['tmp_excel']
5초 대기 후 재확인...
5초 후: ['tmp_excel']
추가 10초 대기 후 재확인...
15초 후: ['tmp_excel']


### trouble shooting 2

In [0]:
print(staging_dir)
import os
print(os.listdir(staging_dir))

/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging


---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
File <command-6937921843551837>, line 3
      1 print(staging_dir)
      2 import os
----> 3 print(os.listdir(staging_dir))

FileNotFoundError: [Errno 2] No such file or directory: '/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging'

In [0]:
manifest = save_bronze_inputs_to_staging(bronze_inputs)

저장 완료: 14개 파일 -> /Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging
manifest.json: /Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging/manifest.json


In [0]:
print(os.listdir(staging_dir))

['terminal_1_schedule_20260626_10.parquet', 'terminal_1_schedule_20260626_11.parquet', 'terminal_2_schedule_20260626_10.parquet', 'terminal_2_schedule_20260626_11.parquet', 'terminal_3_schedule_20260626_10.parquet', 'terminal_3_schedule_20260626_11.parquet', 'terminal_4_schedule_20260626_10.parquet', 'terminal_4_schedule_20260626_11.parquet', 'terminal_5_schedule_20260626_10.parquet', 'terminal_5_schedule_20260626_11.parquet', 'terminal_6_schedule_20260626_10.parquet', 'terminal_6_schedule_20260626_11.parquet', 'terminal_7_schedule_20260626_10.parquet', 'terminal_7_schedule_20260626_11.parquet', 'manifest.json']


### triouble shooting 1

In [0]:
# =====================================================
# 진단: bronze_staging 저장이 왜 안 됐는지 확인
# =====================================================

staging_dir = "/Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging"
local_tmp_dir = "/Workspace/Shared/busan_port_project/01_schedule_pipeline/tmp_excel"

print("=== 1. bronze_staging 폴더 존재 여부 + 내용물 ===")
try:
    contents = dbutils.fs.ls(staging_dir)
    if contents:
        for f in contents:
            print(" -", f.name, f.size)
    else:
        print("폴더는 있지만 내용물이 비어있음")
except Exception as e:
    print("폴더 자체가 없음 또는 접근 에러:", e)

print()
print("=== 2. tmp_excel 폴더 내용물 (parquet 파일이 실제로 만들어졌는지) ===")
try:
    contents = dbutils.fs.ls(local_tmp_dir)
    parquet_files = [f for f in contents if f.name.endswith(".parquet")]
    json_files = [f for f in contents if f.name.endswith(".json")]
    print(f"parquet 파일 개수: {len(parquet_files)}")
    for f in parquet_files:
        print("  -", f.name, f.size)
    print(f"json 파일 개수: {len(json_files)}")
    for f in json_files:
        print("  -", f.name, f.size)
except Exception as e:
    print("에러:", e)

print()
print("=== 3. dbutils.fs.cp 단독 테스트 (한 파일만 직접 복사 시도) ===")
try:
    bronze_inputs  # NameError 직접 유도해서 변수 존재 여부 확실히 체크
except NameError:
    print("bronze_inputs 변수가 정의되어 있지 않습니다. (NameError)")
    print("-> build_bronze_inputs(files) 셀을 이 진단 셀보다 '먼저' 실행했는지 확인하세요.")
else:
    print(f"bronze_inputs 변수 존재함. 길이: {len(bronze_inputs)}")
    try:
        sample_item = bronze_inputs[0]
        sample_df = sample_item["df"].astype(str)
        sample_name = os.path.splitext(sample_item["file"])[0] + "_TEST.parquet"

        local_test_path = f"{local_tmp_dir}/{sample_name}"
        sample_df.to_parquet(local_test_path, index=False)
        print(f"로컬 저장 성공: {local_test_path}")

        dbutils.fs.mkdirs(staging_dir)
        print(f"mkdirs 호출 완료: {staging_dir}")

        final_test_path = f"{staging_dir}/{sample_name}"
        dbutils.fs.cp(f"file:{local_test_path}", final_test_path)
        print(f"cp 성공: {final_test_path}")

        check = dbutils.fs.ls(staging_dir)
        print("복사 후 staging_dir 내용:", [f.name for f in check])

    except Exception as e:
        print("여기서 에러 발생! ↓↓↓")
        print(type(e).__name__, ":", e)

=== 1. bronze_staging 폴더 존재 여부 + 내용물 ===
 - terminal_1_schedule_20260626_10_TEST.parquet 10600

=== 2. tmp_excel 폴더 내용물 (parquet 파일이 실제로 만들어졌는지) ===
parquet 파일 개수: 0
json 파일 개수: 0

=== 3. dbutils.fs.cp 단독 테스트 (한 파일만 직접 복사 시도) ===
bronze_inputs 변수 존재함. 길이: 14
로컬 저장 성공: /Workspace/Shared/busan_port_project/01_schedule_pipeline/tmp_excel/terminal_1_schedule_20260626_10_TEST.parquet
mkdirs 호출 완료: /Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging
cp 성공: /Workspace/Shared/busan_port_project/01_schedule_pipeline/bronze_staging/terminal_1_schedule_20260626_10_TEST.parquet
복사 후 staging_dir 내용: ['terminal_1_schedule_20260626_10_TEST.parquet']


### 검증

In [0]:
# =====================================================
# 검증: 1행=컬럼명 / 2행부터=데이터 가 잘 됐는지 확인
# (bronze 추가컬럼 작업자에게 넘기기 전 최종 점검)
# =====================================================

bronze_inputs = build_bronze_inputs(files)

print(f"총 처리된 파일 개수: {len(bronze_inputs)} (기대값: 14 = 7부두 x 2스냅샷)")
print("=" * 100)

check_results = []

for item in bronze_inputs:
    file_name = item["file"]
    terminal_no = item["terminal_no"]
    df = item["df"]

    columns = df.columns.tolist()

    # 체크 1: 컬럼명이 전부 정수(0,1,2..)면 헤더 분리가 안 된 것 (의심 신호)
    columns_are_all_int = all(isinstance(c, int) for c in columns)

    # 체크 2: 첫 데이터 행 값이 컬럼명과 겹치면(헤더가 데이터로 한 번 더 들어간 경우) 의심 신호
    first_row_values = df.iloc[0].astype(str).tolist() if len(df) > 0 else []
    columns_as_str = [str(c) for c in columns]
    overlap_ratio = (
        sum(1 for v in first_row_values if v in columns_as_str) / len(columns_as_str)
        if columns_as_str else 0
    )
    header_duplicated_in_first_row = overlap_ratio > 0.5  # 절반 이상 겹치면 의심

    status = "OK"
    if columns_are_all_int:
        status = "⚠ 컬럼명이 정수 - 헤더 분리 안 됨"
    elif header_duplicated_in_first_row:
        status = "⚠ 첫 데이터행이 헤더와 중복됨"

    check_results.append({
        "file": file_name,
        "terminal_no": terminal_no,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "column_preview": columns[:5],
        "first_row_preview": first_row_values[:5],
        "status": status,
    })

# 결과를 표로 한눈에 확인
import pandas as pd
summary_df = pd.DataFrame(check_results).sort_values(["terminal_no", "file"])
display(summary_df)

# 문제 있는 파일만 따로 경고 출력
problem_files = [r for r in check_results if r["status"] != "OK"]
if problem_files:
    print(f"\n⚠ 점검 필요한 파일 {len(problem_files)}개:")
    for r in problem_files:
        print(f" - {r['file']} ({r['status']})")
else:
    print("\n✅ 전체 파일 헤더/데이터 분리 정상")

# 부두별 상세 미리보기 (컬럼명 + 첫 2행)도 같이 확인
print("\n" + "=" * 100)
print("부두별 상세 미리보기")
print("=" * 100)
for item in bronze_inputs:
    print(f"\n[{item['file']}] (부두 {item['terminal_no']})")
    print("컬럼:", item["df"].columns.tolist())
    display(item["df"].head(2))

/local_disk0/.ephemeral_nfs/envs/pythonEnv-c86482b7-5caf-4d39-9b19-a0d25dc3c0f9/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/local_disk0/.ephemeral_nfs/envs/pythonEnv-c86482b7-5caf-4d39-9b19-a0d25dc3c0f9/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


총 처리된 파일 개수: 14 (기대값: 14 = 7부두 x 2스냅샷)


file,terminal_no,rows,cols,column_preview,first_row_preview,status
terminal_1_schedule_20260626_10.xls,1,16,15,"List(col_1, col_2, col_3, col_4, col_5)","List(T2(P), ONE, OOSY003, 2612W/2612W, 22 (29) 42)",OK
terminal_1_schedule_20260626_11.xls,1,16,15,"List(col_1, col_2, col_3, col_4, col_5)","List(T2(P), ONE, OOSY003, 2612W/2612W, 22 (29) 42)",OK
terminal_2_schedule_20260626_10.xls,2,34,17,"List(col_1, col_2, col_3, col_4, col_5)","List(1, TEMA MAERSK, TEMM-001/2026, 624E/624E, MAE)",OK
terminal_2_schedule_20260626_11.xls,2,34,17,"List(col_1, col_2, col_3, col_4, col_5)","List(1, TEMA MAERSK, TEMM-001/2026, 624E/624E, MAE)",OK
terminal_3_schedule_20260626_10.xlsx,3,25,18,"List(No, 선석, 항로, 모선항차, 선박명)","List(1, 1B, PNSE, SMMB-2026-0009, SM MUMBAI)",OK
terminal_3_schedule_20260626_11.xlsx,3,25,18,"List(No, 선석, 항로, 모선항차, 선박명)","List(1, 1B, PNSE, SMMB-2026-0009, SM MUMBAI)",OK
terminal_4_schedule_20260626_10.xls,4,21,16,"List(col_1, col_2, col_3, col_4, col_5)","List(T3(S), HMM, HONR003, 0019W/0019W, HMM NURI)",OK
terminal_4_schedule_20260626_11.xls,4,21,16,"List(col_1, col_2, col_3, col_4, col_5)","List(T3(S), HMM, HONR003, 0019W/0019W, HMM NURI)",OK
terminal_5_schedule_20260626_10.xlsx,5,21,9,"List(선석, 선사, 모선항차(선사항차) Head (Bridge) Stern, 선명 (ROUTE), 반입마감시한)","List(1(S), CMA, CAMP004 (0TNIVS1MA/0TNIVS1MA)03 (13) 18, MH PERSEUS(COBOA), 2026/06/23 11:28)",OK
terminal_5_schedule_20260626_11.xlsx,5,21,9,"List(선석, 선사, 모선항차(선사항차) Head (Bridge) Stern, 선명 (ROUTE), 반입마감시한)","List(1(S), CMA, CAMP004 (0TNIVS1MA/0TNIVS1MA)03 (13) 18, MH PERSEUS(COBOA), 2026/06/23 11:28)",OK



✅ 전체 파일 헤더/데이터 분리 정상

부두별 상세 미리보기

[terminal_1_schedule_20260626_10.xls] (부두 1)
컬럼: ['col_1', 'col_2', 'col_3', 'col_4', 'col_5', 'col_6', 'col_7', 'col_8', 'col_9', 'col_10', 'col_11', 'col_12', 'col_13', 'col_14', 'col_15']


col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9,col_10,col_11,col_12,col_13,col_14,col_15
T2(P),ONE,OOSY003,2612W/2612W,22 (29) 42,ONE SERENITY,AN3W,2026-06-22 16:00,2026-06-23 02:10,2026-06-25 13:00,2443,1345,1016,N,DEPARTED
T1(P),MSC,MABX001,UK623A/UK623A,02 (16) 23,MSC ABY X,CHINKE,2026-06-24 00:00,2026-06-24 10:30,2026-06-25 23:31,981,1064,346,N,DEPARTED


...

plvVoy,plvShiftvan,plvNeartml,cdvOperator,atbYn,atdYn,plvStatus,plvEvoyout,plvAtd,cdvName,plvAtb,plvVsl,cct,plvLodvan,plvRoute,plvDisvan,plvYear,plvEvoyin,plvBerth,plvVslvoy,plvQuarantine
001,14,null,MSC,Y,N,Working,FY626A,2026-06-26 17:00,MSC SVEVA,2026-06-25 13:00,MSEV,null,708,AFRICA,1685,2026,FY626A,2(S),MSEV001,검역
001,0,null,MSC,Y,N,Working,GO624S,2026-06-26 13:00,MSC MUNDRA VIII,2026-06-25 19:00,MMUN,null,152,TEMPO,617,2026,GO624S,1(S),MMUN001,null



[terminal_7_schedule_20260626_10.xlsx] (부두 7)
컬럼: ['선석', '선사코드', '모선항차(선사항차)', '모선명(Route)', '반입마감시한', '접안예정일시', '출항예정일시', '작업시작시간', '작업완료시간', 'Head (Bridge) Stern', '작업량\n양하/적하/Shift', '상태']


선석,선사코드,모선항차(선사항차),모선명(Route),반입마감시한,접안예정일시,출항예정일시,작업시작시간,작업완료시간,Head (Bridge) Stern,작업량 양하/적하/Shift,상태
B2(P),ONE,ORGH-001/2026 (007W/007W),GREENHOUSE(VSE),2026-06-23 06:00,2026-06-23 18:00,2026-06-25 06:00,2026-06-23 19:06,2026-06-25 03:39,17 (28+1.56m) 32,1435 / 1127 / 138,Departed
B3(P),DJS,DJFD-018/2026 (0713N/0714S ),DONGJIN FIDES(BKS1),2026-06-24 17:00,2026-06-25 05:00,2026-06-26 06:00,2026-06-25 05:56,2026-06-25 11:02,31 (39+7.50m) 42,88 / 5 / 0,Departed



[terminal_7_schedule_20260626_11.xlsx] (부두 7)
컬럼: ['선석', '선사코드', '모선항차(선사항차)', '모선명(Route)', '반입마감시한', '접안예정일시', '출항예정일시', '작업시작시간', '작업완료시간', 'Head (Bridge) Stern', '작업량\n양하/적하/Shift', '상태']


선석,선사코드,모선항차(선사항차),모선명(Route),반입마감시한,접안예정일시,출항예정일시,작업시작시간,작업완료시간,Head (Bridge) Stern,작업량 양하/적하/Shift,상태
B2(P),ONE,ORGH-001/2026 (007W/007W),GREENHOUSE(VSE),2026-06-23 06:00,2026-06-23 18:00,2026-06-25 06:00,2026-06-23 19:06,2026-06-25 03:39,17 (28+1.56m) 32,1435 / 1127 / 138,Departed
B3(P),DJS,DJFD-018/2026 (0713N/0714S ),DONGJIN FIDES(BKS1),2026-06-24 17:00,2026-06-25 05:00,2026-06-26 06:00,2026-06-25 05:56,2026-06-25 11:02,31 (39+7.50m) 42,88 / 5 / 0,Departed
